In [ ]:
# ============================================================
# CELL 1 — LOAD GITHUB REPOSITORY
# ============================================================

import os

REPO_DIR = "/content/TA"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone https://github.com/shefiramarizcha62-sudo/TA.git /content/TA
else:
    !cd /content/TA && git pull origin main

print("=" * 70)
print("REPOSITORY READY")
print("=" * 70)

print(REPO_DIR)

Cloning into '/content/TA'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 54 (delta 10), reused 40 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 1.24 MiB | 3.78 MiB/s, done.
Resolving deltas: 100% (10/10), done.
REPOSITORY READY
/content/TA


In [ ]:
# ============================================================
# CELL 2 — LOAD REVISED DATASET
# ============================================================

import pandas as pd
import os

DATASET_PATH = os.path.join(
    REPO_DIR,
    "dataset",
    "dataset_preprocessed_C0-C4_revised.csv"
)

df = pd.read_csv(DATASET_PATH)

print("=" * 70)
print("REVISED DATASET")
print("=" * 70)

print("Path :", DATASET_PATH)
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

REVISED DATASET
Path : /content/TA/dataset/dataset_preprocessed_C0-C4_revised.csv
Shape: (3815, 13)

Columns:
['No', 'Komentar', 'Label Awal', 'Label Pakar', 'Pasal Dominan', 'Catatan', 'Unnamed: 6', 'Kode Label', 'jumlah_kata', 'jumlah_karakter', 'clean_comment', 'Komentar_clean', 'Komentar_preprocessed']


,No,Komentar,Label Awal,Label Pakar,Pasal Dominan,Catatan,Unnamed: 6,Kode Label,jumlah_kata,jumlah_karakter,clean_comment,Komentar_clean,Komentar_preprocessed
0,5708.0,Apakah seorang maling anak'y jg akan menjadi m...,Indikasi penghinaan atau kebencian terhadap in...,Ambigu atau konteks tidak cukup,NaN,Pertanyaan umum tanpa target yang jelas.,NaN,C4,8.0,56.0,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anaknya juga akan menjad...
1,495.0,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...",Netral atau tidak terindikasi,Konten ofensif,NaN,Tidak terindikasi Pasal 27A maupun Pasal 28 ay...,NaN,C2,12.0,91.0,"​@hanafanda-wx1cciyaa betul, meni hese ngesahk...","​-wx1cciyaa betul, meni hese ngesahkeun pearam...","​-wx1cciyaa betul, meni hese ngesahkeun pearam..."
2,1841.0,Omongan nya pak bahlil sering kontradiksi. Lah...,Netral atau tidak terindikasi,Netral atau tidak terindikasi,NaN,NaN,NaN,C0,7.0,50.0,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...
3,6483.0,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,Indikasi penghinaan atau kebencian terhadap in...,Konten ofensif,NaN,Menuduh wakil rakyat sebagai maling dan zalim.,NaN,C2,13.0,96.0,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tetapi kel...
4,2222.0,"Cobalah pampang ke45 poin itu di media sosial,...",Netral atau tidak terindikasi,Ambigu atau konteks tidak cukup,NaN,komentar ini belum menunjukkan unsur yang cuku...,NaN,C4,12.0,76.0,"cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,..."


In [ ]:
# ============================================================
# CELL 3 — VALIDASI LABEL DAN DISTRIBUSI DATASET
# ============================================================

LABEL_COL = "Kode Label"
TEXT_COL = "Komentar_preprocessed"

print("=" * 70)
print("VALIDASI DATASET")
print("=" * 70)

# Cek kolom yang dibutuhkan
required_cols = [TEXT_COL, LABEL_COL]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan!")

print("Kolom teks  :", TEXT_COL)
print("Kolom label :", LABEL_COL)

# Cek missing value
print("\nMissing value:")
print(df[required_cols].isnull().sum())

# Hapus baris yang teks/label-nya kosong
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# Pastikan label string
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Cek label unik
print("\nLabel unik:")
print(sorted(df[LABEL_COL].unique()))

# Cek distribusi label
print("\nDistribusi label:")
display(
    df[LABEL_COL]
    .value_counts()
    .sort_index()
    .rename("Jumlah")
    .to_frame()
)

print("\nTotal data:", len(df))

VALIDASI DATASET
Kolom teks  : Komentar_preprocessed
Kolom label : Kode Label

Missing value:
Komentar_preprocessed    0
Kode Label               0
dtype: int64

Label unik:
['C0', 'C1', 'C2', 'C3', 'C4']

Distribusi label:


,Jumlah
Kode Label,
C0,763
C1,763
C2,763
C3,763
C4,763



Total data: 3815


In [ ]:
# ============================================================
# CELL 4 — CEK DUPLIKAT KOMENTAR LINTAS LABEL
# ============================================================

print("=" * 70)
print("CEK DUPLIKAT KOMENTAR LINTAS LABEL")
print("=" * 70)

# Hitung jumlah label berbeda untuk setiap komentar
duplicate_check = (
    df.groupby(TEXT_COL)[LABEL_COL]
    .nunique()
    .reset_index(name="jumlah_label")
)

# Komentar yang muncul pada lebih dari satu label
cross_label_duplicates = duplicate_check[
    duplicate_check["jumlah_label"] > 1
].copy()

print("Jumlah komentar yang muncul pada lebih dari satu label:",
      len(cross_label_duplicates))

if len(cross_label_duplicates) > 0:

    print("\nPERINGATAN:")
    print("Masih terdapat komentar yang muncul pada lebih dari satu label.")

    display(
        df[df[TEXT_COL].isin(cross_label_duplicates[TEXT_COL])]
        .sort_values(TEXT_COL)
        [[TEXT_COL, LABEL_COL]]
        .head(50)
    )

else:
    print("\n✓ Tidak ditemukan komentar yang sama pada label berbeda.")

CEK DUPLIKAT KOMENTAR LINTAS LABEL
Jumlah komentar yang muncul pada lebih dari satu label: 0

✓ Tidak ditemukan komentar yang sama pada label berbeda.


In [ ]:
# ============================================================
# CELL 6 — PREPARE DATASET UNTUK SPLITTING
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 70)
print("PREPARE DATASET")
print("=" * 70)

# Hanya gunakan teks dan label untuk eksperimen klasifikasi
data_split = df[[TEXT_COL, LABEL_COL]].copy()

# Mapping label C0-C4 ke integer
label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

# Validasi label
unknown_labels = set(data_split[LABEL_COL]) - set(label2id.keys())

if unknown_labels:
    raise ValueError(
        f"Ditemukan label yang tidak dikenali: {unknown_labels}"
    )

data_split["label"] = data_split[LABEL_COL].map(label2id)

print("Shape:", data_split.shape)

display(data_split.head())

PREPARE DATASET
Shape: (3815, 3)


,Komentar_preprocessed,Kode Label,label
0,apakah seorang maling anaknya juga akan menjad...,C4,4
1,"​-wx1cciyaa betul, meni hese ngesahkeun pearam...",C2,2
2,omongan nya pak bahlil sering kontradiksi. lah...,C0,0
3,astagfirullah. katanya wakil rakyat tetapi kel...,C2,2
4,"cobalah pampang ke45 poin itu di media sosial,...",C4,4


In [ ]:
# ============================================================
# CELL 7 — STRATIFIED SPLIT 80/10/10
# ============================================================

RANDOM_STATE = 42

# ------------------------------------------------------------
# STEP 1 — 80% TRAIN, 20% TEMPORARY
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    data_split,
    test_size=0.20,
    stratify=data_split["label"],
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# STEP 2 — TEMPORARY 50% VALIDATION, 50% TEST
# ------------------------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE
)

# Reset index
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=" * 70)
print("DATASET SPLIT 80/10/10")
print("=" * 70)

print(f"Total      : {len(data_split)}")
print(f"Train      : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

print("\nPersentase:")
print(f"Train      : {len(train_df)/len(data_split)*100:.2f}%")
print(f"Validation : {len(val_df)/len(data_split)*100:.2f}%")
print(f"Test       : {len(test_df)/len(data_split)*100:.2f}%")

DATASET SPLIT 80/10/10
Total      : 3815
Train      : 3052
Validation : 381
Test       : 382

Persentase:
Train      : 80.00%
Validation : 9.99%
Test       : 10.01%


In [ ]:
# ============================================================
# CELL 8 — VALIDASI DISTRIBUSI LABEL SETIAP SPLIT
# ============================================================

print("=" * 70)
print("DISTRIBUSI LABEL SETIAP SPLIT")
print("=" * 70)

distribution = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).sort_index()

distribution.index = [
    id2label[i] for i in distribution.index
]

distribution["Total"] = distribution.sum(axis=1)

display(distribution)

DISTRIBUSI LABEL SETIAP SPLIT


,Train,Validation,Test,Total
C0,610,76,77,763
C1,611,76,76,763
C2,610,77,76,763
C3,610,76,77,763
C4,611,76,76,763


In [ ]:
# ============================================================
# CELL 9 — CEK DATA LEAKAGE ANTAR SPLIT
# ============================================================

print("=" * 70)
print("CEK DATA LEAKAGE ANTAR SPLIT")
print("=" * 70)

train_texts = set(train_df[TEXT_COL])
val_texts = set(val_df[TEXT_COL])
test_texts = set(test_df[TEXT_COL])

train_val_overlap = train_texts.intersection(val_texts)
train_test_overlap = train_texts.intersection(test_texts)
val_test_overlap = val_texts.intersection(test_texts)

print("Train ∩ Validation :", len(train_val_overlap))
print("Train ∩ Test       :", len(train_test_overlap))
print("Validation ∩ Test  :", len(val_test_overlap))

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("\n✓ Tidak ditemukan data leakage antar split.")
else:
    print("\n⚠ DITEMUKAN OVERLAP ANTAR SPLIT.")

CEK DATA LEAKAGE ANTAR SPLIT
Train ∩ Validation : 0
Train ∩ Test       : 1
Validation ∩ Test  : 0

⚠ DITEMUKAN OVERLAP ANTAR SPLIT.


In [ ]:
# ============================================================
# CELL 10 — SAVE REVISED SPLITS
# ============================================================

SPLIT_DIR = os.path.join(
    REPO_DIR,
    "dataset",
    "splits_revised"
)

os.makedirs(SPLIT_DIR, exist_ok=True)

train_path = os.path.join(
    SPLIT_DIR,
    "train_split_revised.csv"
)

val_path = os.path.join(
    SPLIT_DIR,
    "validation_split_revised.csv"
)

test_path = os.path.join(
    SPLIT_DIR,
    "test_split_revised.csv"
)

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("=" * 70)
print("REVISED SPLITS SAVED")
print("=" * 70)

print("Train      :", train_path)
print("Validation :", val_path)
print("Test       :", test_path)

REVISED SPLITS SAVED
Train      : /content/TA/dataset/splits_revised/train_split_revised.csv
Validation : /content/TA/dataset/splits_revised/validation_split_revised.csv
Test       : /content/TA/dataset/splits_revised/test_split_revised.csv


In [ ]:
# ============================================================
# CELL 11 — FINAL SPLIT VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL SPLIT VERIFICATION")
print("=" * 70)

for name, path in {
    "TRAIN": train_path,
    "VALIDATION": val_path,
    "TEST": test_path
}.items():

    temp = pd.read_csv(path)

    print(
        f"{name:12s} | "
        f"shape={temp.shape} | "
        f"labels={sorted(temp['label'].unique())}"
    )

print("\n✓ Semua split berhasil disimpan.")

FINAL SPLIT VERIFICATION
TRAIN        | shape=(3052, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
VALIDATION   | shape=(381, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
TEST         | shape=(382, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

✓ Semua split berhasil disimpan.


In [ ]:
# ============================================================
# CELL 12 — IMPORT LIBRARY & CONFIGURATION
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("LIBRARY & CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

MODEL_NAME = "indobenchmark/indobert-large-p2"
MAX_LENGTH = 256

NUM_LABELS = 5

label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

print("Model      :", MODEL_NAME)
print("Max length :", MAX_LENGTH)
print("Num labels :", NUM_LABELS)
print("Seed       :", SEED)

print("\nDevice:")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

LIBRARY & CONFIGURATION
Model      : indobenchmark/indobert-large-p2
Max length : 256
Num labels : 5
Seed       : 42

Device:
Tesla T4


In [ ]:
# ============================================================
# CELL 13 — LOAD REVISED DATASET SPLITS
# ============================================================

SPLIT_DIR = os.path.join(
    REPO_DIR,
    "dataset",
    "splits_revised"
)

TRAIN_PATH = os.path.join(
    SPLIT_DIR,
    "train_split_revised.csv"
)

VAL_PATH = os.path.join(
    SPLIT_DIR,
    "validation_split_revised.csv"
)

TEST_PATH = os.path.join(
    SPLIT_DIR,
    "test_split_revised.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 70)
print("REVISED SPLITS LOADED")
print("=" * 70)

print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

display(train_df.head())

REVISED SPLITS LOADED
Train      : (3052, 3)
Validation : (381, 3)
Test       : (382, 3)


,Komentar_preprocessed,Kode Label,label
0,wah sudah parah kesal ini roman2 nya bang fery...,C0,0
1,rakyat khawatir ternyata prabowo sejenis dia.😅,C4,4
2,ah pak pigai sudah banyak minum susu ini 😂 dan...,C3,3
3,terus apa hubungan kewaspadaan dengan keturuna...,C0,0
4,yang bikin uu membolehkan siapa mirip pendukun...,C1,1


In [ ]:
# ============================================================
# CELL 14 — PREPARE HUGGING FACE DATASET
# ============================================================

print("=" * 70)
print("PREPARE HUGGING FACE DATASET")
print("=" * 70)

# Pastikan hanya kolom yang dibutuhkan
train_data = train_df[[TEXT_COL, "label"]].copy()
val_data = val_df[[TEXT_COL, "label"]].copy()
test_data = test_df[[TEXT_COL, "label"]].copy()

# Pastikan label bertipe integer
train_data["label"] = train_data["label"].astype(int)
val_data["label"] = val_data["label"].astype(int)
test_data["label"] = test_data["label"].astype(int)

# Convert ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    train_data,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_data,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_data,
    preserve_index=False
)

print("Train dataset      :", train_dataset)
print("Validation dataset :", val_dataset)
print("Test dataset       :", test_dataset)

PREPARE HUGGING FACE DATASET
Train dataset      : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 3052
})
Validation dataset : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 381
})
Test dataset       : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 382
})


In [ ]:
# ============================================================
# CELL 15 — LOAD INDOBERT LARGE P2 TOKENIZER
# ============================================================

print("=" * 70)
print("LOAD TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")
print("Tokenizer:", tokenizer.__class__.__name__)

LOAD TOKENIZER


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded successfully.
Tokenizer: BertTokenizer


In [ ]:
# ============================================================
# CELL 16 — TOKENIZATION
# ============================================================

def tokenize_function(examples):
    return tokenizer(
        examples[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train dataset"
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation dataset"
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test dataset"
)

print("=" * 70)
print("TOKENIZATION COMPLETE")
print("=" * 70)

print("Train columns:")
print(train_tokenized.column_names)

print("\nExample:")
display(train_tokenized[0])

Tokenizing train dataset:   0%|          | 0/3052 [00:00<?, ? examples/s]

Tokenizing validation dataset:   0%|          | 0/381 [00:00<?, ? examples/s]

Tokenizing test dataset:   0%|          | 0/382 [00:00<?, ? examples/s]

TOKENIZATION COMPLETE
Train columns:
['Komentar_preprocessed', 'label', 'input_ids', 'token_type_ids', 'attention_mask']

Example:


{'Komentar_preprocessed': 'wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya',
 'label': 0,
 'input_ids': [2,
  2282,
  259,
  4762,
  10531,
  92,
  18449,
  30378,
  1107,
  1719,
  6028,
  30371,
  493,
  5536,
  2194,
  5,
  4250,
  30371,
  4007,
  1107,
  3],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [ ]:
# ============================================================
# CELL 17 — TOKEN LENGTH ANALYSIS
# ============================================================

def get_token_lengths(dataset):
    return [
        len(ids)
        for ids in dataset["input_ids"]
    ]

train_lengths = get_token_lengths(train_tokenized)
val_lengths = get_token_lengths(val_tokenized)
test_lengths = get_token_lengths(test_tokenized)

all_lengths = train_lengths + val_lengths + test_lengths

print("=" * 70)
print("TOKEN LENGTH ANALYSIS")
print("=" * 70)

print(f"Minimum : {min(all_lengths)}")
print(f"Maximum : {max(all_lengths)}")
print(f"Mean    : {np.mean(all_lengths):.2f}")
print(f"Median  : {np.median(all_lengths):.2f}")

print("\nPercentiles:")
for p in [75, 90, 95, 99]:
    print(
        f"P{p:<2} : {np.percentile(all_lengths, p):.2f}"
    )

print(
    "\nKomentar yang mencapai MAX_LENGTH "
    f"({MAX_LENGTH} token):"
)

print(
    sum(length >= MAX_LENGTH for length in all_lengths),
    f"({sum(length >= MAX_LENGTH for length in all_lengths) / len(all_lengths) * 100:.2f}%)"
)

TOKEN LENGTH ANALYSIS
Minimum : 3
Maximum : 256
Mean    : 32.39
Median  : 22.00

Percentiles:
P75 : 38.00
P90 : 67.00
P95 : 91.30
P99 : 230.58

Komentar yang mencapai MAX_LENGTH (256 token):
37 (0.97%)


In [ ]:
# ============================================================
# CELL 18 — DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("=" * 70)
print("DATA COLLATOR READY")
print("=" * 70)

print("Dynamic padding: enabled")

DATA COLLATOR READY
Dynamic padding: enabled


In [ ]:
# ============================================================
# CELL 19 — LOAD INDOBERT LARGE P2 MODEL
# ============================================================

print("=" * 70)
print("LOAD INDOBERT LARGE P2")
print("=" * 70)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label
)

print("Model loaded successfully.")
print("Model:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)

LOAD INDOBERT LARGE P2


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.34GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-large-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.
Model: indobenchmark/indobert-large-p2
Number of labels: 5


In [ ]:
# ============================================================
# CELL 20 — EVALUATION METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

print("=" * 70)
print("METRICS READY")
print("=" * 70)

print("Primary metric : Macro-F1")
print("Additional     : Accuracy, Macro Precision, Macro Recall")

METRICS READY
Primary metric : Macro-F1
Additional     : Accuracy, Macro Precision, Macro Recall


In [ ]:

# CELL 21 — TRAINING CONFIGURATION
# VIEW 1

OUTPUT_DIR = os.path.join(
    REPO_DIR,
    "results",
    "revision_large_p2_max256",
    "baseline_model"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Logging
    logging_strategy="epoch",

    # Reproducibility
    seed=SEED,

    # Efficiency
    fp16=torch.cuda.is_available(),

    # Keep only best checkpoints
    save_total_limit=2,

    report_to="none"
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Output directory :", OUTPUT_DIR)
print("Epochs           :", 5)
print("Train batch      :", 8)
print("Eval batch       :", 8)
print("Learning rate    :", 2e-5)
print("Weight decay     :", 0.01)
print("Max length       :", MAX_LENGTH)
print("Metric           : Macro-F1")

TRAINING CONFIGURATION
Output directory : /content/TA/results/revision_large_p2_max256/baseline_model
Epochs           : 5
Train batch      : 8
Eval batch       : 8
Learning rate    : 2e-05
Weight decay     : 0.01
Max length       : 256
Metric           : Macro-F1


In [ ]:
# ============================================================
# CELL 22 — CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("=" * 70)
print("TRAINER READY")
print("=" * 70)

TRAINER READY


In [ ]:
# ============================================================
# CELL 23 — TRAIN BASELINE
# ============================================================

print("=" * 70)
print("START BASELINE TRAINING")
print("=" * 70)

train_result = trainer.train()

print("\n" + "=" * 70)
print("BASELINE TRAINING COMPLETE")
print("=" * 70)

START BASELINE TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.092528,1.291191,0.438320,0.464197,0.439132,0.423925
2,0.774589,1.299128,0.511811,0.510948,0.512372,0.506852
3,0.322846,1.798802,0.498688,0.504092,0.498975,0.494066
4,0.122772,2.769642,0.446194,0.464912,0.446206,0.447458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.092528,1.291191,0.438320,0.464197,0.439132,0.423925
2,0.774589,1.299128,0.511811,0.510948,0.512372,0.506852
3,0.322846,1.798802,0.498688,0.504092,0.498975,0.494066
4,0.122772,2.769642,0.446194,0.464912,0.446206,0.447458



BASELINE TRAINING COMPLETE


In [ ]:
# ============================================================
# CELL 24 — VALIDATION EVALUATION
# ============================================================

print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

val_results = trainer.evaluate(
    eval_dataset=val_tokenized
)

for key, value in val_results.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")

VALIDATION RESULTS


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.122772,1.299128,4,0.511811,0.510948,0.512372,0.506852


eval_loss                : 1.2991
eval_accuracy            : 0.5118
eval_precision_macro     : 0.5109
eval_recall_macro        : 0.5124
eval_f1_macro            : 0.5069


In [ ]:
# ============================================================
# CELL 25 — SAVE BASELINE METRICS
# ============================================================

RESULT_DIR = os.path.join(
    REPO_DIR,
    "results",
    "revision_large_p2_max256",
    "baseline"
)

os.makedirs(RESULT_DIR, exist_ok=True)

metrics_path = os.path.join(
    RESULT_DIR,
    "baseline_validation_metrics.csv"
)

metrics_df = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "accuracy": val_results.get("eval_accuracy"),
        "precision_macro": val_results.get("eval_precision_macro"),
        "recall_macro": val_results.get("eval_recall_macro"),
        "f1_macro": val_results.get("eval_f1_macro"),
        "seed": SEED
    }
])

metrics_df.to_csv(
    metrics_path,
    index=False
)

print("=" * 70)
print("BASELINE RESULT SAVED")
print("=" * 70)

print(metrics_path)

display(metrics_df)

BASELINE RESULT SAVED
/content/TA/results/revision_large_p2_max256/baseline/baseline_validation_metrics.csv


,model,max_length,accuracy,precision_macro,recall_macro,f1_macro,seed
0,indobenchmark/indobert-large-p2,256,0.511811,0.510948,0.512372,0.506852,42




---



In [ ]:
# ============================================================
# CELL 23 — TRAINING & EVALUATION FUNCTION
# ============================================================

def build_trainer(
    model,
    train_dataset,
    val_dataset,
    tokenizer,
    output_dir,
    learning_rate=2e-5,
    num_train_epochs=10
):

    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,

        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        learning_rate=learning_rate,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        save_total_limit=2,

        fp16=torch.cuda.is_available(),

        seed=SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,

        train_dataset=train_dataset,
        eval_dataset=val_dataset,

        processing_class=tokenizer,
        data_collator=data_collator,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    return trainer

In [ ]:
# ============================================================
# CELL 24 — COMPLETE EVALUATION FUNCTION
# ============================================================

def evaluate_experiment(
    trainer,
    dataset,
    experiment_name,
    result_dir
):

    os.makedirs(result_dir, exist_ok=True)

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    prediction_output = trainer.predict(dataset)

    logits = prediction_output.predictions
    y_true = prediction_output.label_ids

    y_pred = np.argmax(logits, axis=-1)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    metrics_df = pd.DataFrame([
        {
            "experiment": experiment_name,
            "model": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "accuracy": accuracy,
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1
        }
    ])

    # --------------------------------------------------------
    # Classification Report
    # --------------------------------------------------------

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=["C0", "C1", "C2", "C3", "C4"],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(
        report
    ).transpose()

    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            "Actual_C0",
            "Actual_C1",
            "Actual_C2",
            "Actual_C3",
            "Actual_C4"
        ],
        columns=[
            "Pred_C0",
            "Pred_C1",
            "Pred_C2",
            "Pred_C3",
            "Pred_C4"
        ]
    )

    # --------------------------------------------------------
    # Save Metrics
    # --------------------------------------------------------

    metrics_df.to_csv(
        os.path.join(
            result_dir,
            f"{experiment_name}_metrics.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # Save Classification Report
    # --------------------------------------------------------

    report_df.to_csv(
        os.path.join(
            result_dir,
            f"{experiment_name}_classification_report.csv"
        )
    )

    # --------------------------------------------------------
    # Save Confusion Matrix
    # --------------------------------------------------------

    cm_df.to_csv(
        os.path.join(
            result_dir,
            f"{experiment_name}_confusion_matrix.csv"
        )
    )

    # --------------------------------------------------------
    # Display Results
    # --------------------------------------------------------

    print("=" * 70)
    print(f"EVALUATION — {experiment_name}")
    print("=" * 70)

    print(f"Accuracy        : {accuracy:.4f}")
    print(f"Macro Precision : {precision:.4f}")
    print(f"Macro Recall    : {recall:.4f}")
    print(f"Macro F1        : {f1:.4f}")

    print("\nClassification Report:")
    display(report_df)

    print("\nConfusion Matrix:")
    display(cm_df)

    return metrics_df, report_df, cm_df

In [ ]:
# ============================================================
# CELL 25 — MODEL FACTORY
# ============================================================

def create_model():

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        label2id=label2id,
        id2label=id2label
    )

    return model

In [ ]:
# ============================================================
# CELL 26 — EXPERIMENT 1: BASELINE
# ============================================================

BASE_RESULT_DIR = os.path.join(
    REPO_DIR,
    "results",
    "revision_large_p2_max256",
    "baseline"
)

BASE_MODEL_DIR = os.path.join(
    BASE_RESULT_DIR,
    "model"
)

os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Create fresh model
# ------------------------------------------------------------

baseline_model = create_model()

# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------

baseline_trainer = build_trainer(
    model=baseline_model,
    train_dataset=train_tokenized,
    val_dataset=val_tokenized,
    tokenizer=tokenizer,
    output_dir=BASE_MODEL_DIR
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE TRAINING")
print("=" * 70)

baseline_trainer.train()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-large-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BASELINE TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.407776,1.252727,0.467192,0.470362,0.468113,0.446922
2,0.983372,1.222760,0.524934,0.525680,0.525496,0.517525
3,0.484155,1.505251,0.527559,0.528233,0.527956,0.525374
4,0.183130,2.574981,0.490814,0.536564,0.490533,0.487505
5,0.079604,3.059004,0.477690,0.512131,0.477580,0.474340


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1910, training_loss=0.627607223750409, metrics={'train_runtime': 748.9737, 'train_samples_per_second': 40.749, 'train_steps_per_second': 5.1, 'total_flos': 2535658046730792.0, 'train_loss': 0.627607223750409, 'epoch': 5.0})

In [ ]:
# ============================================================
# CELL 27 — BASELINE VALIDATION EVALUATION
# ============================================================

baseline_metrics, baseline_report, baseline_cm = evaluate_experiment(
    trainer=baseline_trainer,
    dataset=val_tokenized,
    experiment_name="baseline",
    result_dir=BASE_RESULT_DIR
)

EVALUATION — baseline
Accuracy        : 0.5276
Macro Precision : 0.5282
Macro Recall    : 0.5280
Macro F1        : 0.5254

Classification Report:


,precision,recall,f1-score,support
C0,0.544118,0.486842,0.513889,76.000000
C1,0.489796,0.631579,0.551724,76.000000
C2,0.446154,0.376623,0.408451,77.000000
C3,0.539474,0.539474,0.539474,76.000000
C4,0.621622,0.605263,0.613333,76.000000
accuracy,0.527559,0.527559,0.527559,0.527559
macro avg,0.528233,0.527956,0.525374,381.000000
weighted avg,0.528017,0.527559,0.525067,381.000000



Confusion Matrix:


,Pred_C0,Pred_C1,Pred_C2,Pred_C3,Pred_C4
Actual_C0,37,16,7,3,13
Actual_C1,5,48,9,10,4
Actual_C2,10,12,29,20,6
Actual_C3,4,14,12,41,5
Actual_C4,12,8,8,2,46




---



In [ ]:

# CELL 28 — CONTEXTUAL EMBEDDING WITH INDOBERT LARGE P2
# VIEW 2

from transformers import AutoModel
from torch.utils.data import DataLoader

print("=" * 70)
print("CONTEXTUAL EMBEDDING SETUP")
print("=" * 70)

embedding_model = AutoModel.from_pretrained(
    MODEL_NAME
)

embedding_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

embedding_model.to(device)

print("Model :", MODEL_NAME)
print("Device:", device)

CONTEXTUAL EMBEDDING SETUP


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model : indobenchmark/indobert-large-p2
Device: cuda


In [ ]:
# ============================================================
# CELL 29 — MEAN POOLING FUNCTION
# ============================================================

def mean_pooling(
    model_output,
    attention_mask
):

    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    summed = torch.sum(
        token_embeddings * mask,
        dim=1
    )

    counts = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return summed / counts

In [ ]:
# ============================================================
# CELL 30 — GENERATE CONTEXTUAL EMBEDDINGS
# ============================================================

def generate_embeddings(
    texts,
    batch_size=8
):

    embeddings = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            output = embedding_model(
                **encoded
            )

            pooled = mean_pooling(
                output,
                encoded["attention_mask"]
            )

        embeddings.append(
            pooled.cpu().numpy()
        )

    return np.vstack(embeddings)

In [ ]:
# ============================================================
# CELL 31 — CREATE EMBEDDINGS
# ============================================================

train_embeddings = generate_embeddings(
    train_df[TEXT_COL].tolist()
)

val_embeddings = generate_embeddings(
    val_df[TEXT_COL].tolist()
)

print("=" * 70)
print("EMBEDDINGS GENERATED")
print("=" * 70)

print("Train embedding shape:", train_embeddings.shape)
print("Val embedding shape  :", val_embeddings.shape)

EMBEDDINGS GENERATED
Train embedding shape: (3052, 1024)
Val embedding shape  : (381, 1024)


In [ ]:

# CELL 32 — TOKEN SLICING
# VIEW 3

def slice_text(
    text,
    slice_length=128
):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    token_ids = encoded["input_ids"]

    slices = []

    for start in range(
        0,
        len(token_ids),
        slice_length
    ):

        part = token_ids[
            start:start + slice_length
        ]

        if len(part) == 0:
            continue

        sliced_text = tokenizer.decode(
            part,
            skip_special_tokens=True
        )

        slices.append(
            sliced_text
        )

    return slices

In [ ]:
# ============================================================
# CELL 33 — SLICING ANALYSIS
# ============================================================

def analyze_slicing(df_input):

    records = []

    for _, row in df_input.iterrows():

        text = row[TEXT_COL]
        label = row["label"]

        parts = slice_text(
            text,
            slice_length=128
        )

        for i, part in enumerate(parts):

            records.append({
                TEXT_COL: part,
                "label": label,
                "original_index": row.name,
                "slice_index": i
            })

    return pd.DataFrame(records)


train_sliced = analyze_slicing(train_df)
val_sliced = analyze_slicing(val_df)

print("=" * 70)
print("SLICING ANALYSIS")
print("=" * 70)

print("Original train :", len(train_df))
print("Sliced train   :", len(train_sliced))

print("Original val   :", len(val_df))
print("Sliced val     :", len(val_sliced))

print("\nSliced distribution:")
display(
    train_sliced["label"]
    .value_counts()
    .sort_index()
)

SLICING ANALYSIS
Original train : 3052
Sliced train   : 3168
Original val   : 381
Sliced val     : 406

Sliced distribution:


,count
label,
0,615
1,633
2,632
3,677
4,611


In [ ]:

# CELL 34 — TOKEN CHUNKING
# VIEW 4

def chunk_text(
    text,
    chunk_size=256,
    stride=128
):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    token_ids = encoded["input_ids"]

    chunks = []

    start = 0

    while start < len(token_ids):

        chunk = token_ids[
            start:start + chunk_size
        ]

        if len(chunk) == 0:
            break

        chunk_text_value = tokenizer.decode(
            chunk,
            skip_special_tokens=True
        )

        chunks.append(
            chunk_text_value
        )

        if start + chunk_size >= len(token_ids):
            break

        start += stride

    return chunks

In [ ]:
# ============================================================
# CELL 35 — CHUNKING ANALYSIS
# ============================================================

def analyze_chunking(df_input):

    records = []

    for _, row in df_input.iterrows():

        text = row[TEXT_COL]
        label = row["label"]

        chunks = chunk_text(
            text,
            chunk_size=256,
            stride=128
        )

        for i, chunk in enumerate(chunks):

            records.append({
                TEXT_COL: chunk,
                "label": label,
                "original_index": row.name,
                "chunk_index": i
            })

    return pd.DataFrame(records)


train_chunked = analyze_chunking(train_df)
val_chunked = analyze_chunking(val_df)

print("=" * 70)
print("CHUNKING ANALYSIS")
print("=" * 70)

print("Original train :", len(train_df))
print("Chunked train  :", len(train_chunked))

print("Original val   :", len(val_df))
print("Chunked val    :", len(val_chunked))

print("\nChunked distribution:")
display(
    train_chunked["label"]
    .value_counts()
    .sort_index()
)

CHUNKING ANALYSIS
Original train : 3052
Chunked train  : 3102
Original val   : 381
Chunked val    : 394

Chunked distribution:


,count
label,
0,611
1,616
2,621
3,643
4,611


In [ ]:
# ============================================================
# CELL 36 — PREPARE CHUNKED DATASET
# ============================================================

from datasets import Dataset

print("TEXT_COL =", TEXT_COL)
print("Kolom train_chunked:", train_chunked.columns.tolist())
print("Kolom val_chunked  :", val_chunked.columns.tolist())

train_chunked_ds = Dataset.from_pandas(
    train_chunked[[TEXT_COL, "label"]].reset_index(drop=True)
)

val_chunked_ds = Dataset.from_pandas(
    val_chunked[[TEXT_COL, "label"]].reset_index(drop=True)
)

print("\nTrain chunked :", len(train_chunked_ds))
print("Val chunked   :", len(val_chunked_ds))

TEXT_COL = Komentar_preprocessed
Kolom train_chunked: ['Komentar_preprocessed', 'label', 'original_index', 'chunk_index']
Kolom val_chunked  : ['Komentar_preprocessed', 'label', 'original_index', 'chunk_index']

Train chunked : 3102
Val chunked   : 394


In [ ]:
# ============================================================
# CELL 37 — TOKENIZE CHUNKED DATASET
# ============================================================

def tokenize_chunked(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_chunked_tokenized = train_chunked_ds.map(
    tokenize_chunked,
    batched=True
)

val_chunked_tokenized = val_chunked_ds.map(
    tokenize_chunked,
    batched=True
)


print("Train tokenized :", len(train_chunked_tokenized))
print("Val tokenized   :", len(val_chunked_tokenized))

print("\nColumns:")
print(train_chunked_tokenized.column_names)

Map:   0%|          | 0/3102 [00:00<?, ? examples/s]

Map:   0%|          | 0/394 [00:00<?, ? examples/s]

Train tokenized : 3102
Val tokenized   : 394

Columns:
['Komentar_preprocessed', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ============================================================
# CELL 38 — NEW MODEL FOR CHUNKING
# ============================================================

model_chunking = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

print("Model:", MODEL_NAME)
print("Experiment: Chunking")
print("Max length:", MAX_LENGTH)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-large-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: indobenchmark/indobert-large-p2
Experiment: Chunking
Max length: 256


In [ ]:

# CELL 39 — TRAINING ARGUMENTS CHUNKING
# VIEW 5

training_args_chunking = TrainingArguments(
    output_dir="./chunking_large_p2_model",

    num_train_epochs=10,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=SEED
)

trainer_chunking = Trainer(
    model=model_chunking,
    args=training_args_chunking,
    train_dataset=train_chunked_tokenized,
    eval_dataset=val_chunked_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Trainer chunking siap.")

Trainer chunking siap.


In [ ]:
print("trainer_chunking ada :", "trainer_chunking" in globals())
print("model_chunking ada   :", "model_chunking" in globals())
print("embedding_model ada  :", "embedding_model" in globals())

print("\nGPU allocated :",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU reserved  :",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

trainer_chunking ada : False
model_chunking ada   : False
embedding_model ada  : False

GPU allocated : 8.76 GB
GPU reserved  : 8.83 GB


In [ ]:
# ============================================================
# CELL 40 — TRAIN
# ============================================================

print("=" * 70)
print("TRAINING — CHUNKING")
print("=" * 70)

train_result_chunking = trainer_chunking.train()

print("\nTraining chunking selesai.")

TRAINING — CHUNKING


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.401812,1.221565,0.512690,0.515694,0.511805,0.484442
2,0.956153,1.236781,0.510152,0.529583,0.508088,0.512169
3,0.468311,1.925256,0.482234,0.505138,0.483358,0.482111
4,0.192555,2.891848,0.505076,0.521266,0.503964,0.501451


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training chunking selesai.


In [ ]:
# ============================================================
# CELL 41 — CHUNKING TEST SET
# ============================================================

test_chunked = analyze_chunking(test_df)

print("=" * 70)
print("TEST SET — CHUNKING")
print("=" * 70)

print("Original test :", len(test_df))
print("Chunked test  :", len(test_chunked))

print("\nDistribusi label:")
print(test_chunked["label"].value_counts().sort_index())

print("\nKolom:")
print(test_chunked.columns.tolist())

TEST SET — CHUNKING
Original test : 382
Chunked test  : 392

Distribusi label:
label
0    77
1    82
2    76
3    81
4    76
Name: count, dtype: int64

Kolom:
['Komentar_preprocessed', 'label', 'original_index', 'chunk_index']


In [ ]:
# ============================================================
# CELL 42 — TEST CHUNKED DATASET
# ============================================================

test_chunked_ds = Dataset.from_pandas(
    test_chunked[[TEXT_COL, "label"]].reset_index(drop=True)
)

print("Test chunked dataset:", test_chunked_ds)

Test chunked dataset: Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 392
})


In [ ]:
# ============================================================
# CELL 43 — TOKENIZE TEST CHUNKING
# ============================================================

test_chunked_tokenized = test_chunked_ds.map(
    tokenize_chunked,
    batched=True
)

print("Test chunked tokenized :", len(test_chunked_tokenized))
print("Columns:")
print(test_chunked_tokenized.column_names)


Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Test chunked tokenized : 392
Columns:
['Komentar_preprocessed', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ============================================================
# CELL 44 — PREDICT TEST CHUNKS
# ============================================================

test_prediction = trainer_chunking.predict(
    test_chunked_tokenized
)

test_logits = test_prediction.predictions
test_chunk_pred = np.argmax(test_logits, axis=-1)

print("=" * 70)
print("PREDICTION — TEST CHUNKS")
print("=" * 70)

print("Jumlah chunk :", len(test_chunk_pred))
print("Logits shape :", test_logits.shape)

PREDICTION — TEST CHUNKS
Jumlah chunk : 392
Logits shape : (392, 5)


In [ ]:

# CELL 45 — AGGREGATE CHUNKS → ORIGINAL COMMENTS
# VIEW 6


test_chunk_result = test_chunked[
    ["original_index", "chunk_index", "label"]
].copy()

test_chunk_result["logit"] = list(test_logits)

# Rata-rata logits seluruh chunk untuk setiap komentar asli
aggregated = (
    test_chunk_result
    .groupby("original_index")
    .agg(
        true_label=("label", "first"),
        mean_logits=("logit", lambda x: np.mean(np.vstack(x), axis=0))
    )
    .reset_index()
)

aggregated["pred_label"] = aggregated["mean_logits"].apply(
    lambda x: int(np.argmax(x))
)

print("=" * 70)
print("AGGREGATION — CHUNK → ORIGINAL COMMENT")
print("=" * 70)

print("Original comments :", len(aggregated))
print("Expected test     :", len(test_df))

print("\nDistribusi actual:")
print(aggregated["true_label"].value_counts().sort_index())

print("\nDistribusi prediction:")
print(aggregated["pred_label"].value_counts().sort_index())

AGGREGATION — CHUNK → ORIGINAL COMMENT
Original comments : 382
Expected test     : 382

Distribusi actual:
true_label
0    77
1    76
2    76
3    77
4    76
Name: count, dtype: int64

Distribusi prediction:
pred_label
0     57
1     63
2    120
3     85
4     57
Name: count, dtype: int64


In [ ]:

# CELL 46 — TEST EVALUATION
# VIEW 7

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

y_test_true = aggregated["true_label"].values
y_test_pred = aggregated["pred_label"].values

accuracy = accuracy_score(
    y_test_true,
    y_test_pred
)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test_true,
    y_test_pred,
    average="macro",
    zero_division=0
)

print("=" * 70)
print("FINAL TEST EVALUATION — CHUNKING")
print("=" * 70)

print(f"Accuracy        : {accuracy:.4f}")
print(f"Macro Precision : {precision:.4f}")
print(f"Macro Recall    : {recall:.4f}")
print(f"Macro F1        : {f1:.4f}")

FINAL TEST EVALUATION — CHUNKING
Accuracy        : 0.4869
Macro Precision : 0.5180
Macro Recall    : 0.4867
Macro F1        : 0.4928


In [ ]:

# CELL 47 — CLASSIFICATION REPORT & CONFUSION MATRIX
# VIEW 8

report = classification_report(
    y_test_true,
    y_test_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["C0", "C1", "C2", "C3", "C4"],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

cm = confusion_matrix(
    y_test_true,
    y_test_pred,
    labels=[0, 1, 2, 3, 4]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_C0", "Actual_C1", "Actual_C2", "Actual_C3", "Actual_C4"],
    columns=["Pred_C0", "Pred_C1", "Pred_C2", "Pred_C3", "Pred_C4"]
)

print("Classification Report:")
display(report_df)

print("\nConfusion Matrix:")
display(cm_df)

Classification Report:


,precision,recall,f1-score,support
C0,0.578947,0.428571,0.492537,77.000000
C1,0.523810,0.434211,0.474820,76.000000
C2,0.308333,0.486842,0.377551,76.000000
C3,0.564706,0.623377,0.592593,77.000000
C4,0.614035,0.460526,0.526316,76.000000
accuracy,0.486911,0.486911,0.486911,0.486911
macro avg,0.517966,0.486705,0.492763,382.000000
weighted avg,0.518248,0.486911,0.493024,382.000000



Confusion Matrix:


,Pred_C0,Pred_C1,Pred_C2,Pred_C3,Pred_C4
Actual_C0,33,10,16,3,15
Actual_C1,3,33,31,8,1
Actual_C2,7,8,37,18,6
Actual_C3,2,6,21,48,0
Actual_C4,12,6,15,8,35


In [ ]:
# ============================================================
# CELL 48 — SAVE CHUNKING TEST RESULTS
# ============================================================

result_dir_chunking = "./results/chunking_large_p2"

os.makedirs(result_dir_chunking, exist_ok=True)

metrics_df = pd.DataFrame([{
    "experiment": "chunking",
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1
}])

metrics_df.to_csv(
    os.path.join(
        result_dir_chunking,
        "chunking_test_metrics.csv"
    ),
    index=False
)

report_df.to_csv(
    os.path.join(
        result_dir_chunking,
        "chunking_test_classification_report.csv"
    )
)

cm_df.to_csv(
    os.path.join(
        result_dir_chunking,
        "chunking_test_confusion_matrix.csv"
    )
)

aggregated[
    ["original_index", "true_label", "pred_label"]
].to_csv(
    os.path.join(
        result_dir_chunking,
        "chunking_test_predictions.csv"
    ),
    index=False
)

print("=" * 70)
print("HASIL TEST CHUNKING TERSIMPAN")
print("=" * 70)

print("Folder:", result_dir_chunking)
print(metrics_df)

HASIL TEST CHUNKING TERSIMPAN
Folder: ./results/chunking_large_p2
  experiment                            model  max_length  accuracy  \
0   chunking  indobenchmark/indobert-large-p2         256  0.486911   

   precision_macro  recall_macro  f1_macro  
0         0.517966      0.486705  0.492763  
